In [26]:
def revrse_numbs(num):
    s_num = str(num)
    for nums in (s_num):
        print(nums[::-1])
num =[1234]
revrse_numbs(num)

[
1
2
3
4
]


In [ ]:
wget
https: // www.robots.ox.ac.uk / ~vgg / data / pets / data / images.tar.gz & & tar - xf
images.tar.gz
! wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz && tar -xf annotations.tar.gz
import cv2

cv2.imread('images/Abyssinian_1.jpg')
cv2.imread("./annotations/trimaps/Abyssinian_1.png")
annot = cv2.imread("./annotations/trimaps/Abyssinian_1.png")
np.max(annot), np.min(annot)
plt.imshow(annot * 70)
import os

root_img = './images'
root_label = './annotations/trimaps'

imgs = []
labels = []
for img_name in os.listdir(root_img):
    if img_name.endswith(".jpg"):
        name = img_name.split(".")[0]
        label_name = name + ".png"
        label_path = os.path.join(root_label, label_name)
        if os.path.exists(label_path):
            imgs.append(os.path.join(root_img, img_name))
            labels.append(os.path.join(root_label, label_name))

len(imgs), len(labels)
imgs[1970], labels[1970]
cv2.imread(imgs[1970])


class seg_dataset(torch.utils.data.Dataset):
    def __init__(self, root_img, root_label, transform=None, target_transform=None, classes_num=None):
        self.root_img = root_img
        self.root_label = root_label
        self.transform = transform
        self.target_transform = target_transform
        self.classes_num = classes_num

        self.imgs = []
        self.labels = []

        NoneType = type(None)

        for img_name in os.listdir(self.root_img):
            if img_name.endswith(".jpg"):
                name = img_name.split(".")[0]
                label_name = name + ".png"

                label_path = os.path.join(self.root_label, label_name)
                img_path = os.path.join(self.root_img, img_name)
                if os.path.exists(label_path) and os.path.exists(img_path):

                    img = cv2.imread(img_path)
                    label = cv2.imread(label_path)
                    if isinstance(img, NoneType) or isinstance(label, NoneType):
                        print(img_path, label_path)
                    else:
                        self.imgs.append(img_path)
                        self.labels.append(label_path)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx])
        label = cv2.imread(self.labels[idx])

        img = self.transform(img)
        label = np.array(self.target_transform(label))
        w, h, c = label.shape
        annot = torch.zeros(size=(self.classes_num, w, h), dtype=torch.float)

        # print(annot.shape, label.shape, img.shape)
        for i in range(self.classes_num):
            annot[i, :, :] = torch.tensor(label[:, :, 0] == i + 1, dtype=torch.int)

        return img, annot


import os

transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize(256),
    v2.CenterCrop((224, 224)),
    v2.ToTensor(),
])

target_transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize(256),
    v2.CenterCrop((224, 224)),
])

dataset = seg_dataset(root_img='./images', root_label='./annotations/trimaps', transform=transform,
                      target_transform=target_transform, classes_num=3)

train_set, test_set = torch.utils.data.random_split(dataset,
                                                    [int(len(dataset) * 0.7), len(dataset) - int(len(dataset) * 0.7)])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

for imgs, labels in train_loader:
    break
idx = 3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(imgs[idx].permute(1, 2, 0))
ax1.set_title("Original")

ax2.imshow(labels[idx].permute(1, 2, 0)[:, :, 2] * 70, cmap='gray')
ax2.set_title("Reconstructed!")

labels[3].permute(1, 2, 0)[:, :, 0].shape
torch.max(labels[3].permute(1, 2, 0)[:, :, 2])
labels.shape


class seg_transposed(nn.Module):
    def __init__(self, classes_num=1):
        super(seg_transposed, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),  # 3*224*224 -> 16*224*224
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16*224*224 -> 16*112*112

            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=1),  # 16*112*112 -> 16*112*112
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16*112*112 -> 16*56*56

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),  # 16*56*56 -> 32*56*56
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32*56*56 -> 32*28*28

            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1),  # 32*28*28 -> 32*28*28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32*28*28 -> 32*14*14

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),  # 32*14*14 -> 64*14*14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64*14*14 -> 64*7*7

            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),  # 64*7*7 -> 64*7*7
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=2, stride=2),  # 64*7*7 -> 64*14*14
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=2, stride=2),  # 64*14*14 -> 64*28*28
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=2, stride=2),  # 64*28*28 -> 32*56*56
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2, stride=2),  # 32*56*56 -> 16*112*112
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=16, out_channels=classes_num, kernel_size=2, stride=2),
            # 16*112*112 -> classes_num*224*224
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)

        return x


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = seg_transposed(classes_num=3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

model = model.to(device)
criterion = criterion.to(device)

for epoch in range(10):
    losses = []
    for imgs, labels in tqdm(train_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print("epoch: ", epoch, "loss: ", np.mean(losses))
# video = cv2.VideoCapture("/sfhhaoiryhasd.mp4")
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame = transform(frame)
    out = model(frame)

    if torch.argmax(out) == 0:  # if classification is equal to class number 1
        with open("./alerts.txt", 'w') as f:
            f.write(f"Person detected! {time.time.datetime()}")